## encodings: PANDA x UNI2-h / Virchow2

PANDA ships as raw gigapixel WSIs, not pre-cut patches, so this notebook does two things the other two don't need:

1. **Tile** a sample of slides into tissue patches using the Otsu foreground-detection    approach prototyped in `exploration/explore_panda.ipynb`.
2. **Embed** those patches with UNI2-h and Virchow2, and save both the patch-level    encodings and a slide-level mean-pooled encoding (one vector per slide, useful    for probing the slide-level `isup_grade` label directly).

Caveat: patches are sampled at level-0 (native resolution) without checking each slide's microns-per-pixel, so magnification isn't calibrated across slides/scanners the way a production pipeline would want -- fine for a first pass, worth revisiting if results look off between Radboud and Karolinska slides.

In [ ]:
!du -sh /home/shared/data/panda

In [ ]:
import os

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/home/shared/.cache/huggingface/hub"
os.environ["TRANSFORMERS_CACHE"] = (
    "/home/shared/.cache/huggingface/hub"  # deprecated alias, harmless to set
)

In [ ]:
!pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys
import pyrootutils

# VS Code's Jupyter kernel starts in the launch dir, not the notebook's folder,
# so an upward search from cwd can miss the repo root -- search from the notebook
# file itself (VS Code sets __vsc_ipynb_file__) and fall back to cwd otherwise.
search_from = globals().get("__vsc_ipynb_file__", ".")
root = pyrootutils.setup_root(search_from=search_from, indicator=".project-root", pythonpath=False)
sys.path.append(str(root / "src"))
from vfm_encoders import load_uni2, load_virchow2, embed_images

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import openslide
from pathlib import Path
from tqdm.auto import tqdm

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the model pages, then set `HF_TOKEN` (or leave unset for an interactive prompt).

In [ ]:
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load slide-level metadata

Prefers `train_subset.csv` (the stratified subset from the download notebook) if it exists on disk, otherwise falls back to the full `train.csv` filtered to slides actually present locally.

In [ ]:
panda_base_dir = "/home/shared/data/panda/"
train_images_dir = panda_base_dir + "train_images/"

csv_path = Path(panda_base_dir) / "train_subset.csv"
if not csv_path.exists():
    csv_path = Path(panda_base_dir) / "train.csv"

train_df = (
    pd.read_csv(csv_path)
    .assign(
        filepath=lambda x: x["image_id"].apply(lambda y: str(Path(train_images_dir) / f"{y}.tiff"))
    )
    .assign(exists=lambda x: x["filepath"].apply(lambda y: Path(y).exists()))
    .loc[lambda x: x["exists"]]
)
print(f"Slides available: {len(train_df)}")
train_df["isup_grade"].value_counts().sort_index()

### 2. Tiling: sample tissue patches from a slide

Same Otsu foreground mask + random-in-foreground sampling used interactively in `explore_panda.ipynb`, wrapped into a function so it can run over many slides.

In [ ]:
def sample_tissue_patches(
    slide_path: str,
    patch_size: int = 224,
    n_patches: int = 16,
    seed: int = 42,
):
    """Returns a list of up to n_patches RGB PIL patches (level-0 resolution)
    sampled from the tissue (foreground) region of the slide, plus their (x, y)
    top-left coordinates in level-0 pixel space.
    """
    slide = openslide.OpenSlide(slide_path)
    thumbnail = slide.get_thumbnail((512, 512))
    thumb_arr = np.array(thumbnail.convert("L"))

    _, foreground_mask = cv2.threshold(thumb_arr, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))

    fg_ys, fg_xs = np.nonzero(foreground_mask)
    if len(fg_xs) == 0:
        slide.close()
        return [], []

    rng = np.random.default_rng(seed)
    n_sample = min(n_patches, len(fg_xs))
    sample_idx = rng.choice(len(fg_xs), size=n_sample, replace=False)

    scale_x = slide.dimensions[0] / thumb_arr.shape[1]
    scale_y = slide.dimensions[1] / thumb_arr.shape[0]

    patches, coords = [], []
    for idx in sample_idx:
        cx = int(fg_xs[idx] * scale_x)
        cy = int(fg_ys[idx] * scale_y)
        x0 = int(np.clip(cx - patch_size // 2, 0, slide.dimensions[0] - patch_size))
        y0 = int(np.clip(cy - patch_size // 2, 0, slide.dimensions[1] - patch_size))

        patch = slide.read_region((x0, y0), level=0, size=(patch_size, patch_size)).convert("RGB")
        patches.append(patch)
        coords.append((x0, y0))

    slide.close()
    return patches, coords

### 3. Tile the selected slides

`N_SLIDES` slides x `N_PATCHES_PER_SLIDE` patches each -- keep this modest for a first pass, it's easy to scale up once the pipeline is confirmed working end to end.

In [ ]:
N_SLIDES = 100
N_PATCHES_PER_SLIDE = 1000
PATCH_SIZE = 224

slides_to_tile = train_df.sample(n=min(N_SLIDES, len(train_df)), random_state=42)

patches = []
patch_records = []
for i, (_, row) in tqdm(
    enumerate(slides_to_tile.iterrows()), total=len(slides_to_tile), desc="tiling slides"
):
    slide_patches, coords = sample_tissue_patches(
        row["filepath"], patch_size=PATCH_SIZE, n_patches=N_PATCHES_PER_SLIDE
    )
    for patch, (x0, y0) in zip(slide_patches, coords):
        patches.append(patch)
        patch_records.append(
            {
                "image_id": row["image_id"],
                "isup_grade": row["isup_grade"],
                "data_provider": row["data_provider"],
                "x0": x0,
                "y0": y0,
            }
        )

    with open("log.txt", "w") as f:
        f.write(f"{i}\n")

patch_df = pd.DataFrame(patch_records)
print(f"Tiled {len(patch_df)} patches from {patch_df['image_id'].nunique()} slides")

### 4. Load the encoders

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}
for enc in encoders.values():
    n_params = sum(p.numel() for p in enc.model.parameters())
    print(f"{enc.name:10s} embed_dim={enc.embed_dim:5d}  params={n_params / 1e6:.0f}M")

### 5. Extract patch encodings

In [ ]:
encodings = {enc.name: embed_images(enc, patches, device=device) for enc in encoders.values()}

### 6. Save patch-level and slide-level (mean-pooled) encodings

Slide-level features average all patches from the same slide into a single vector -- the natural feature to probe against the slide-level `isup_grade` label.

In [ ]:
out_dir = Path(panda_base_dir) / "encodings"
out_dir.mkdir(parents=True, exist_ok=True)

suffix = f"n{patch_df['image_id'].nunique()}slides_{len(patch_df)}patches"
patch_df.to_csv(out_dir / f"patch_metadata_{suffix}.csv", index=False)

slide_labels = train_df.set_index("image_id")[["isup_grade", "data_provider"]]

for model_name, feats in encodings.items():
    np.save(out_dir / f"{model_name}_patch_{suffix}.npy", feats)

    slide_ids = patch_df["image_id"].values
    slide_feats = pd.DataFrame(feats, index=slide_ids).groupby(level=0).mean()
    slide_feats = slide_feats.join(slide_labels, how="left")
    slide_feats.to_csv(out_dir / f"{model_name}_slide_{suffix}.csv")

    print(f"Saved {model_name}: {feats.shape[0]} patches, {len(slide_feats)} slides")

### 7. Sanity check

PCA of the slide-level (mean-pooled) encodings, colored by ISUP grade -- higher grades should drift away from grade 0 if the encoder is picking up tumor morphology.

In [ ]:
slide_feats

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

fig, axs = plt.subplots(1, len(encoders), figsize=(6 * len(encoders), 5))
for ax, model_name in zip(axs, encodings):
    slide_feats = pd.read_csv(out_dir / f"{model_name}_slide_{suffix}.csv", index_col=0)
    feat_cols = [c for c in slide_feats.columns if c not in ("isup_grade", "data_provider")]
    coords = PCA(n_components=2, random_state=42).fit_transform(slide_feats[feat_cols])

    sc = sns.scatterplot(
        x=coords[:, 0],
        y=coords[:, 1],
        hue=slide_feats["isup_grade"],
        palette="RdYlGn_r",
        style=slide_feats["data_provider"],
        s=100,
        legend=ax == axs[-1],
        ax=ax,
    )
    ax.set_title(f"{model_name} -- slide-level, by ISUP grade")
    # plt.colorbar(sc, ax=ax)

sns.move_legend(ax, bbox_to_anchor=(1, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

fig, axs = plt.subplots(1, len(encoders), figsize=(6 * len(encoders), 5))
for ax, model_name in zip(axs, encodings):
    slide_feats = pd.read_csv(out_dir / f"{model_name}_slide_{suffix}.csv", index_col=0)
    feat_cols = [c for c in slide_feats.columns if c not in ("isup_grade", "data_provider")]
    coords = PCA(n_components=2, random_state=42).fit_transform(slide_feats[feat_cols])

    sc = sns.scatterplot(
        x=coords[:, 0],
        y=coords[:, 1],
        hue=slide_feats["data_provider"],
        palette="tab10",
        s=40,
        ax=ax,
    )
    ax.set_title(f"{model_name} -- slide-level, by ISUP grade")
    # plt.colorbar(sc, ax=ax)

plt.tight_layout()
plt.show()